This notebook contains code on collecting population from Global Human Settlement Layer (GHSL) dataset for the generated grids and then calculating the population value for each 100m x 100m grid. The developed code relies on google earth engine for processing and google cloud for intermediate storage of data.

In [6]:
import ee
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds
import os
import numpy as np
import logging
from multiprocess import Pool
import glob
from functools import partial

### data collection

In [ ]:
# set relative path for grid files
grid_path = 'data/*.parquet'

# create a list of all parquet grid files from the specified directory
grids_list = [parquet for parquet in glob.glob(grid_path)]

# authenticate and Initialize Google Earth Engine
ee.Authenticate()
ee.Initialize()

In [ ]:
# GHSL population data
dataset = ee.Image('JRC/GHSL/P2023A/GHS_POP/2020')
population = dataset.select('population_count')

In [ ]:
# function to define and start export tasks
def export_population(grid_path):
    """
    Export slope images for each grid using Google Earth Engine.
    
    Args:
    - grid_path (str): Path to the grid file.
    """
    grid_number = grid_path.split('_')[-1].split('.')[0]
    img_name = f'pop_100m_{grid_number}.tif'

    grid_data = gpd.read_parquet(grid_path)
    extent = grid_data.to_crs('epsg:4326').total_bounds

    ee_bounds = ee.Geometry.Polygon([
        [[extent[0], extent[1]],
         [extent[0], extent[3]],
         [extent[2], extent[3]],
         [extent[2], extent[1]]]
    ])

#     export_params = {
#         'image': population,
#         'bucket': 'cog-bucket-test',    # your bucket name
#         'description': f'pop_100m_{grid_number}',
#         'fileNamePrefix': f'pop_100m_{grid_number}',
#         'scale': 100,
#         'region': ee_bounds.getInfo()['coordinates'],
#         'crs': 'EPSG:3035',
#         'fileFormat': 'GeoTIFF',
#         'formatOptions': {'cloudOptimized': True},
#         'maxPixels': 1e12,
#     }

#     task = ee.batch.Export.image.toCloudStorage(**export_params)
    
    task = ee.batch.Export.image.toDrive(
        image=population,
        description=f'pop_100m_{grid_number}',
        folder='walkability_temp',     # Google Drive folder name
        fileNamePrefix=f'pop_100m_{grid_number}',
        scale=100,
        region=ee_bounds,
        crs='EPSG:3035',
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True},
        maxPixels=1e12
    )
    
    task.start()

In [ ]:
# sequential
for elem in grids_list:
    export_population(elem)

### calculate population for 100m x 100m grids

In [7]:
# set the environment variable for Google Cloud credentials
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'path/to/your/credentials.json'

# list to hold the paths of the parquet files
grids_list = []
for parquet_file in glob.glob('data/*.parquet'):
    grids_list.append(parquet_file)

# configure logging (recommended if you monitor processing over a lot of files)
log_path = 'logs/population.log'

# ensure log directory exists
log_dir = os.path.dirname(log_path)
if not os.path.exists(log_dir):
    os.makedirs(log_dir)
    
logging.basicConfig(filename=log_path, level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s', force=True)

In [2]:
import os
import io
from pathlib import Path
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
FOLDER_ID = 'walkability/pop'
DEST_DIR = Path('/rds/user/trpb2/hpc-work/walkability/walkability-obct/data/test')


from google_auth_oauthlib.flow import Flow

def authenticate():
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = Flow.from_client_secrets_file(
                'credentials.json',
                scopes=SCOPES,
                redirect_uri='urn:ietf:wg:oauth:2.0:oob'  # <-- this is the key change
            )
            auth_url, _ = flow.authorization_url(prompt='consent')
            print(f"\nGo to this URL in your browser:\n\n{auth_url}\n")
            code = input("Paste the authorisation code here: ")
            flow.fetch_token(code=code)
            creds = flow.credentials
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return creds


def download_folder(service, folder_id, dest_dir):
    dest_dir.mkdir(parents=True, exist_ok=True)
    
    results = service.files().list(
        q=f"'{folder_id}' in parents and trashed = false",
        fields="files(id, name, mimeType)"
    ).execute()
    
    for item in results.get('files', []):
        name, fid, mime = item['name'], item['id'], item['mimeType']
        
        if mime == 'application/vnd.google-apps.folder':
            # Recurse into subfolders
            download_folder(service, fid, dest_dir / name)
        elif mime.startswith('application/vnd.google-apps'):
            # Skip Google Docs/Sheets/Slides (export separately if needed)
            print(f"Skipping Google Workspace file: {name}")
        else:
            dest_path = dest_dir / name
            print(f"Downloading: {name}")
            request = service.files().get_media(fileId=fid)
            with open(dest_path, 'wb') as f:
                downloader = MediaIoBaseDownload(f, request)
                done = False
                while not done:
                    _, done = downloader.next_chunk()


if __name__ == '__main__':
    creds = authenticate()
    service = build('drive', 'v3', credentials=creds)
    download_folder(service, FOLDER_ID, DEST_DIR)
    print("Done.")


Go to this URL in your browser:

https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=95770466819-17vrsi0eecuvkpisj3pklcsc4qg8fqta.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=ju3fIw8OrnMctGDN7N1RhzHzcJl1SM&code_challenge=glJKFBCokLzwv5kBw4m5tzs4ogw2UXkUMG_5f3PLgr0&code_challenge_method=S256&prompt=consent&access_type=offline

Paste the authorisation code here: 4/1AUn6Hd9boy4FPVGS1yP8yJv9rCTTalXcwJfyQt-Ss5_kuO9EEYyncR_oMTE


HttpError: <HttpError 404 when requesting https://www.googleapis.com/drive/v3/files?q=%27walkability%2Fpop%27+in+parents+and+trashed+%3D+false&fields=files%28id%2C+name%2C+mimeType%29&alt=json returned "File not found: .". Details: "[{'message': 'File not found: .', 'domain': 'global', 'reason': 'notFound', 'location': 'fileId', 'locationType': 'parameter'}]">

In [8]:
def calculate_population(row, src):
    """
    Calculate the population value for a single row (geometry) from a GeoDataFrame.
    
    Args:
    - row (GeoSeries): A GeoDataFrame row representing a polygon.
    - src (rasterio.io.DatasetReader): Open raster source to read data from.
    
    Returns:
    - float: Population value from the raster.
    """
    left, bottom, right, top = row.geometry.bounds
    window = from_bounds(left, bottom, right, top, src.transform)
    data = src.read(window=window)
    img = data[0]

    if img.shape != (1, 1):
        logging.info(f'More than one pixel in the window')
        return np.nan  # return NaN when the expected data isn't found

    return img[0, 0]

def process_grid(grid_path):
    """
    Process a single grid file to calculate and update population data.
    
    Args:
    - grid_path (str): Path to the grid file.
    """
    try:
        grid_number = grid_path.split('_')[-1].split('.')[0]
        grid_gdf = gpd.read_parquet(grid_path)

        if 'population' in grid_gdf.columns:
            logging.info(f'Skipping grid {grid_number} as population already calculated')
            return

        #cog_path = f'./eurostat_grid/grids_100/ghsl_pop/pop_100m_{grid_number}.tif'
        cog_path = f'/rds/user/trpb2/hpc-work/walkability/walkability-obct/data/pop/pop_100m_{grid_number}.tif'
        logging.info(f'Started processing grid {grid_number} at {cog_path}')

        with rasterio.open(cog_path) as src:
            if grid_gdf.crs != src.crs:
                grid_gdf = grid_gdf.to_crs(src.crs)
            calculate_with_src = partial(calculate_population, src=src)
            grid_gdf['population'] = grid_gdf.apply(lambda row: calculate_with_src(row), axis=1)

        grid_gdf.to_parquet(grid_path)
        logging.info(f'Successfully processed grid {grid_path}')
    except Exception as e:
        logging.error(f'Error processing grid {grid_path}: {e}')

In [9]:
# sequential
# for elem in grids_list:
#     process_grid(elem)

# # parallel processing setup
num_processes = 5
with Pool(processes=num_processes) as pool:
    pool.map(process_grid, grids_list)

In [2]:
import geopandas as gpd

gdf = gpd.read_parquet("data/grids_100_608.parquet")
gdf.head()

,geometry,grid_100000_id,slope,population,dou
0,"POLYGON ((3600000 3200000, 3600100 3200000, 36...",608,1.058738,146.612640,30
1,"POLYGON ((3600000 3200100, 3600100 3200100, 36...",608,1.476905,121.456001,30
2,"POLYGON ((3600000 3200200, 3600100 3200200, 36...",608,1.962831,26.572197,30
3,"POLYGON ((3600000 3200300, 3600100 3200300, 36...",608,2.380930,0.343318,30
4,"POLYGON ((3600000 3200400, 3600100 3200400, 36...",608,2.580068,0.000000,30
